# Bag-of-Words MaxRL

Sample script for zero-step-sampling MaxRL on the bag-of-words dataset.

The policy is a Gaussian around the model's scalar prediction, m_theta(z | x) = Normal(f_theta(x), sigma^2). MaxRL weights use the Gaussian likelihood of the noisy target under each rollout.

In [ ]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.maxrl import BagOfWordsMaxRLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:0")

## Configure

In [ ]:
config = BagOfWordsMaxRLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-maxrl-example",
    corr=0.5,
    aux_words_ratio=0.5,
    train_epochs=2,
    num_rollouts_per_sample=2,
    gaussian_stdev=1.0,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")
print(f"Rollouts/sample: {config.num_rollouts_per_sample}")
print(f"Canonical degree: {config.degree}")
print(f"Policy stdev:     {config.gaussian_stdev}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

In [ ]:
state.run_training()

## Results

Training saves:
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics.
2. Validation parquet containing per-row ground-truth and target.

In [ ]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

In [ ]:
analysis = BagOfWordsAnalysisConfig.from_studies({"example": config.study_folder})
epoch_axis = pl.col("epoch").alias("epoch")

analysis.xy_plots([
    (epoch_axis, rsq_expr(split="train", y="ground_truth"), None),
    (epoch_axis, rsq_expr(split="val", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="train", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="val", y="ground_truth"), None),
])

In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()